# Zuzai work-type labels — fine-tuning on Colab

**How to run:** Runtime → Change runtime type → **A100 GPU** → **Runtime → Run all**.

**Google Drive space:** checkpoints (model + optimizer, ≈ 2 GB each for DeBERTa-v3-base) are written to the Colab VM disk
(`/content/ckpt`), **not** to Drive. Only the results go to Drive (≈ 0.5 MB per fold). The cleanup cell also removes
leftovers from earlier runs (embedding chunks, old checkpoints).
*Files deleted from a mounted Drive go to the Drive trash and still count against the quota — empty it at
https://drive.google.com/drive/trash.*

**If Colab disconnects:** reconnect and **Run all** again. Finished folds are skipped. If the VM itself was kept, the
running fold resumes from its last checkpoint; if the VM was replaced, that one fold restarts (≈ 7–15 min on A100).

**Results:** `My Drive/zuzai-hw/exports/ft_<model>.zip` (refreshed after every fold). Download them into the project's
`data/` folder or run `uv run python src/import_colab.py ~/Downloads/ft_*.zip` on the Mac.

In [ ]:
# ---- Config (edit only if needed) ----
# Set after the benchmark (notebooks/colab_ft_benchmark.ipynb). Done so far: qwen3-0.6b-lora folds 0–4.
JOBS = [
    ("qwen3-8b-lora", [0, 1, 2, 3, 4]),
]
EPOCHS = 3
EXTRA = ""               # fastest setting that fits, from the benchmark (e.g. "--no-gc --batch-size 4 --accum 4")
CLASS_WEIGHT = "inv"     # inv (as Zuzai's DistilBERT) | sqrt_inv | none
REPO = "https://github.com/ShayHummel/work-type-labels.git"
BRANCH = "solution"
OUT = "/content/drive/MyDrive/zuzai-hw"   # results (small)
CKPT_ROOT = "/content/ckpt"               # checkpoints on the VM disk (large, not on Drive)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
os.makedirs(OUT, exist_ok=True)
if not os.path.exists("/content/work-type-labels"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/work-type-labels"], check=True)
else:
    subprocess.run(["git", "-C", "/content/work-type-labels", "pull", "-q"], check=True)
%cd /content/work-type-labels
!git log --oneline -1

In [ ]:
# ---- Free Drive space: remove leftovers from earlier runs (keeps all results and exports) ----
import glob, shutil
print("before:")
!du -sh {OUT}/* 2>/dev/null
for d in glob.glob(f"{OUT}/emb/*/chunks") + glob.glob(f"{OUT}/ft/*/fold*/ckpt"):
    print("removing", d); shutil.rmtree(d, ignore_errors=True)
print("after:")
!du -sh {OUT}/* 2>/dev/null
print("Now empty the Drive trash: https://drive.google.com/drive/trash")

In [ ]:
!pip install -q "transformers==5.18.0" "accelerate==1.15.0" "peft==0.21.2" "sentencepiece==0.2.2" "protobuf"
!pip uninstall -y -q torchao   # Colab ships torchao 0.10; peft rejects < 0.16 and we do not use it
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
for model, folds in JOBS:
    print(f"===== {model} folds {folds} =====", flush=True)
    !python src/finetune.py --model {model} --folds {" ".join(map(str, folds))} --out {OUT} --ckpt-root {CKPT_ROOT} --epochs {EPOCHS} --class-weight {CLASS_WEIGHT} {EXTRA}

In [ ]:
!ls -lh {OUT}/exports
!grep -h DONE {OUT}/logs/ft_*.log